In [ ]:
import torch
import torch.nn as nn
import torchvision
import albumentations as A
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from torchvision.models import segmentation as seg_models
from torchvision import transforms as T
from torchvision.transforms import functional as F
# from unet import UNet
from tqdm import tqdm
# from torchsummary import summary
import time
import os
import numpy as np
import cv2
from PIL import Image
from typing import List, Dict, Tuple, Union, Final
import sys
sys.path.append('../coco_eval/custom_utils')
from json_parser import create_dataset_class, CellMaskDataset

## Configurations
### Model parameters

In [ ]:
# Dataset class names to class IDs map, the class IDs will be returned by the model
# if more than 1 class is being predicted, make sure to use a higher class ID for the class that is supposed to be 
# detected in case two objects of different classes overlay
CLASS_NAMES_TO_CLASS_IDS_MAP: Dict[str, int] = {'cell-adhered': 1}
PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES: Final[float] = 0.1

# number of classes (including background)
NUM_CLASSES: Final[int] = len(CLASS_NAMES_TO_CLASS_IDS_MAP) + 1
Z_STACK_OVERLAY_MAP: Dict[str, str] = {'White_dz-16': 'red', 'White_dz0': 'green', 'White_dz16': 'blue'}

# model input image large/small-side sizes
MODEL_INPUT_MAX_SIZE: Final[int] = 1024
MODEL_INPUT_MIN_SIZE: Final[int] = 800 

### Training parameters

In [ ]:
# training batch size
# this should be at least 2 as the DeepLab model Batch norm require at least 2
BATCH_SIZE: Final[int] = 2
# learning rate
LEARNING_RATE: Final[float] = 1e-3
# number of training epochs
NUM_EPOCHS = 10
# learning rate decay steps, a value of 0 means One-cycle LR scheduler should be used
LR_DECAY_STEPS = 0

### Train/Test image and annotation folders

In [ ]:
DATASET_BASE_PATH: str = '/home/labuser/workspaces/mehdi/semantic_segmentation/data/20250227_preadipocytes-adhered_4x_caged'

MODEL_PATH = 'checkpoints'
if not os.path.exists(MODEL_PATH):
    os.mkdir(MODEL_PATH)

## Data Model 
### Dataset class

In [ ]:
def build_semantic_mask(data_sample: dict, class_ids_of_interest: List[int]):

    img_height, img_width = data_sample['image'].shape[:2]
    
    # semantic mask in full image resolution
    # we are using np.uint8, hence only 255 segments (which is fine)    
    semantic_mask: np.ndarray = np.zeros((img_height, img_width), np.uint8) # zero is the background always

    for class_id in class_ids_of_interest:
        # go over the classes in the provided order to ensure the order of overlapping objects
        # we create separate semantic masks for all the objects of the same class first, and then overlay them
        objs_of_interest_df: pd.DataFrame = data_sample['annotations'][data_sample['annotations']['label'] == class_id]
        semantic_mask_this_class = np.zeros((img_height, img_width), np.uint8)
        for row_idx, row in objs_of_interest_df.iterrows():
            xmin, ymin, xmax, ymax = row[['xtl', 'ytl', 'xbr', 'ybr']]
            # no need to check the validity 
            if xmin >= xmax or ymin >= ymax:
                continue
                
            # update the semantic mask
            instance_mask: np.ndarray = data_sample['masks'][row_idx] * class_id 
            # only update the non-zero areas, otherwise, we may remove parts of
            # the semantic mask from other instances (note that these are all from the same class)
            semantic_mask_this_class[ymin:ymax, xmin:xmax][instance_mask > 0] = instance_mask[instance_mask > 0]
        # now combine the semantic masks for different classes
        # since we are going over the class IDs in a given order, cage masks will be replaced by cells, and 
        # cell masks will be replaced by beads
        semantic_mask = np.maximum(semantic_mask, semantic_mask_this_class)
    return semantic_mask

class SemanticMaskDataset(Dataset):
    
    def __init__(self, 
                 dataset_path: str,
                 class_names_to_class_ids_map: Dict[str, int],
                 train: bool,
                 mean: np.array, 
                 std: np.array,
                 model_input_size: Tuple[int, int],
                 transform = None,
                ):

        self.dataset = create_dataset_class(
            dataset_path=dataset_path, 
            class_names_to_class_ids_map=class_names_to_class_ids_map, 
            train=train,
            bit_depth=8,
            min_object_diameter=0.0,
            percentage_to_expand_bbox_boundaries=PERCENTAGE_TO_EXPAND_BBOX_BOUNDARIES, 
            max_images_to_consider_for_each_annotation=0, # we are generating a z-stack overlaid image, specified in the FL channel ID map
            only_use_best_focus_image=True, 
            max_larger_side=max(model_input_size), 
            max_smaller_side=min(model_input_size), 
            normalize_img=False,
            fl_channel_id_to_color_map_for_overlay=Z_STACK_OVERLAY_MAP
        )
        
        self.class_names_to_class_ids_map = class_names_to_class_ids_map
        self.transform = transform
        self.mean: np.array = mean
        self.std: np.array = std
        self.model_input_size: Tuple[int, int] = model_input_size
    
    def __len__(self) -> int:
        return len(self.dataset)
    
    def __getitem__(self, idx: int) -> (torch.Tensor, torch.Tensor):

        sample = self.dataset[idx]
        
        image: np.ndarray = sample['image']
        if len(image.shape) < 3:
            # the model expects a 3 channel image, 
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

        semantic_mask: np.ndarray = build_semantic_mask(
            data_sample=sample, 
            class_ids_of_interest=list(self.class_names_to_class_ids_map.values())
        )
 
        if self.transform is not None:
            augmented = self.transform(image=image, mask=semantic_mask)
            image = augmented['image']
            semantic_mask = augmented['mask']

        # convert the image (numpy array) to a torch Tensor and normalize it
        convert_normalize_t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        image_tensor: torch.Tensor = convert_normalize_t(image)
        mask_tensor: torch.Tensor = torch.from_numpy(semantic_mask).long()
        
        return image_tensor, mask_tensor


from pycocotools import mask as coco_mask_util
import pickle
import pandas as pd
class SemanticMaskDatasetFroCroppedImages(torch.utils.data.Dataset):
    def __init__(
        self, 
        images_path: str, 
        masks_path: str, 
        class_ids_map: Dict[int, int],
        mean: np.array, 
        std: np.array,
        model_input_size: Tuple[int, int], # (width, height) format
        transform=None
    ) -> None:
        self.images_path = images_path
        self.masks_path = masks_path
        self.mean: np.array = mean
        self.std: np.array = std
        self.transform = transform
        # load all images and masks
        # the assumption is the image and its mask annotation use the same name
        self.imgs = list(sorted(os.listdir(images_path)))
        self.masks = list(sorted(os.listdir(masks_path)))
        self.class_ids_map = class_ids_map
        self.model_input_size = model_input_size

        if len(self.imgs) != len(self.masks):
            print("[ERROR]: The list of images and masks are not consistent")
            return
        
        for i, img_filename in enumerate(self.imgs):
            # drop the image/mask filename extension 
            # (anything after the last '.' in the filename is considered as extension)
            img_name = ".".join(img_filename.strip().split('.')[:-1])
            mask_name = ".".join(self.masks[i].strip().split('.')[:-1])
            if img_name != mask_name:
                print("[ERROR]: Inconsistent mask file :{} found for image file: {}".format(mask_name, img_name))
     

    def __getitem__(self, idx: int):
        # load images and masks
        img_path = os.path.join(self.images_path, self.imgs[idx])
        mask_path = os.path.join(self.masks_path, self.masks[idx])
        # read the image, do not change the format
        # depending on the set color_depth, the values will be in [0, 2^color_depth - 1]
        
        img = Image.open(img_path)

        image_width, image_height = img.size
        
        boxes: List[List] = []
        labels: List[int] = []
        masks: List[np.ndarray] = []
            
        # load the annotations
        filehandler = open(mask_path, 'rb')
        annots = pickle.load(filehandler)
        filehandler.close()
            
        for record in annots['annotations']:
            xmin, ymin, xmax, ymax = record['bbox']
            # no need to check the validity 
            if xmin >= xmax or ymin >= ymax:
                continue

            label: int = int(record['category_id'])
            
            if label not in self.class_ids_map:
                continue
            
            labels.append(self.class_ids_map[label])
            masks.append(coco_mask_util.decode(record['segmentation']))
            boxes.append([xmin, ymin, xmax, ymax])
        
        labels: np.ndarray = np.array(labels)
        boxes: np.ndarray = np.array(boxes)

        image: np.ndarray = np.array(img)
        if len(image.shape) < 3:
            # the model expects a 3 channel image, 
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            
        sample = {
            'image': image,
            'annotations': pd.DataFrame({'xtl': boxes[:, 0] ,
                                               'ytl': boxes[:, 1], 
                                               'xbr': boxes[:, 2], 
                                               'ybr': boxes[:, 3], 
                                               'label': labels}
                                             ),
            'masks': masks}
        
        semantic_mask: np.ndarray = build_semantic_mask(
            data_sample=sample, 
            class_ids_of_interest=list(self.class_ids_map.values())
        )

        max_factor: float = max(image_height, image_width) / float(max(self.model_input_size)) 
        min_factor: float = min(image_height, image_width) / float(min(self.model_input_size))
        factor: float = max(max_factor, min_factor)

        if factor > 1:
            # resize to make sure the image and the mask are within the model input size, however, this may have 
            # negative impact on training as we are downsizing the image and the objects
            # it is expected the parsed images to have the same size and the model input size
            print("[WARN]: The train/test image is resized while keeping the aspect ratio. This may be unwanted!")
            print("[WARN]: The parsed images should be of the same size as the model input.")
            print(f"[WARN]: The image size is {(image_width, image_heigh)} and the model input size is {self.model_input_size}.")

            if image_height > image_width:
                if factor == max_factor:
                    # factor = image_height / max(self.model_input_size) -> new width = image_width / image_height * max(self.model_input_size)
                    # image_width / image_height < 1 -> new width is less than max(self.model_input_size)
                    # min_factor < max_factor - > image_width < min(self.model_input_size) -> new width < min(self.model_input_size)
                    target_size = Tuple[int, int] = (int(image_width / factor), max(self.model_input_size))
                else:
                    target_size = Tuple[int, int] = (min(self.model_input_size), int(image_height / factor))     
            else:
                if factor == max_factor:
                    target_size = Tuple[int, int] = (max(self.model_input_size), int(image_height / factor))
                else:
                    target_size = Tuple[int, int] = (int(image_width / factor), min(self.model_input_size))   
                                
            image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
            semantic_mask = cv2.resize(image, target_size, interpolation=cv2.INTER_NEAREST)
 
        if self.transform is not None:
            augmented = self.transform(image=image, mask=semantic_mask)
            image = augmented['image']
            semantic_mask = augmented['mask']

        # convert the image (numpy array) to a torch Tensor and normalize it
        convert_normalize_t = T.Compose([T.ToTensor(), T.Normalize(self.mean, self.std)])
        image_tensor: torch.Tensor = convert_normalize_t(image)
        mask_tensor: torch.Tensor = torch.from_numpy(semantic_mask).long()
        
        return image_tensor, mask_tensor

    def __len__(self):
        return len(self.imgs)

### Image and segmentation mask transforms
Here, we use albumentations package that takes both image and annotations to apply the transformations on them. It takes and returns numpy arrays. 

In [ ]:
train_transform = A.Compose(
    [A.HorizontalFlip(), 
     A.VerticalFlip(), 
     A.GridDistortion(p=0.2), 
     # A.RandomRotate90(),
     A.Perspective(p=0.2),
     A.RandomBrightnessContrast(p=0.2),
     A.CoarseDropout(
         num_holes_range=(1, int(0.02 * MODEL_INPUT_MAX_SIZE * MODEL_INPUT_MIN_SIZE)),
         hole_height_range=(1, 1),
         hole_width_range=(1, 1), 
         p=0.2
     ),
     A.GaussianBlur(sigma_limit=(0, 2.0), p = 0.2),
     A.AdditiveNoise(
         noise_type='gaussian', 
         noise_params={'mean_range': (0, 0), 'std_range': (0, 0.04)}, 
         p=0.2, 
         spatial_mode='per_pixel'
     )]
)

### Datasets and dataloaders

In [ ]:
# datasets
train_dataset = SemanticMaskDataset(
    dataset_path=DATASET_BASE_PATH,
    class_names_to_class_ids_map=CLASS_NAMES_TO_CLASS_IDS_MAP,
    train=True,
    mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225],
    model_input_size=(MODEL_INPUT_MIN_SIZE, MODEL_INPUT_MAX_SIZE),
    transform=train_transform
)

test_dataset = SemanticMaskDataset(
    dataset_path=DATASET_BASE_PATH,
    class_names_to_class_ids_map=CLASS_NAMES_TO_CLASS_IDS_MAP,
    train=False,
    mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225],
    model_input_size=(MODEL_INPUT_MIN_SIZE, MODEL_INPUT_MAX_SIZE),
    transform=None
)

# dataloaders
# drop_last is set to True to avoid passing a data with batch size of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)     

In [ ]:
# cropped images datasets
PARSED_DATASET_BASE_PATH = '/home/labuser/workspaces/mehdi/semantic_segmentation/data/20250227_preadipocytes-adhered_4x_caged_z_stack_overlaid_16u_6_class'

train_dataset = SemanticMaskDatasetFroCroppedImages(
    images_path=os.path.join(PARSED_DATASET_BASE_PATH, 'z_stack_images', 'train'),
    masks_path=os.path.join(PARSED_DATASET_BASE_PATH, 'masks', 'train'), 
    class_ids_map={6:1}, # nucleus
    mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225],
    model_input_size=(MODEL_INPUT_MIN_SIZE, MODEL_INPUT_MAX_SIZE),
    transform=train_transform
)

test_dataset = SemanticMaskDatasetFroCroppedImages(
    images_path=os.path.join(PARSED_DATASET_BASE_PATH, 'z_stack_images', 'test'),
    masks_path=os.path.join(PARSED_DATASET_BASE_PATH, 'masks', 'test'), 
    class_ids_map={6:1}, # nucleus
    mean=[0.485, 0.456, 0.406], 
    std=[0.229, 0.224, 0.225],
    model_input_size=(MODEL_INPUT_MIN_SIZE, MODEL_INPUT_MAX_SIZE),
    transform = None
)

# dataloaders
# drop_last is set to True to avoid passing a data with batch size of 1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)     

### Visualization

In [ ]:
from sem_seg_utils import show_sample

In [ ]:
img = show_sample(3, train_dataset)
Image.fromarray(img[:, :, ::-1])

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT) # Load net
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)

# Change final layer to 2 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

## Training
### Optimizer, LR scheduler and loss function

In [ ]:
# construct an optimizer
# Adam optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr = LEARNING_RATE)

print(f"Adam Optimizer is configured for {NUM_EPOCHS} epochs")

print(f"Initial learning rate is set to {LEARNING_RATE}")
if LR_DECAY_STEPS < 1:
    print(f"One-Cyle LR scheduler is configured for {NUM_EPOCHS} with {len(train_loader)} steps per epoch")
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, 
                                                       max_lr=LEARNING_RATE, 
                                                       epochs=NUM_EPOCHS,
                                                       steps_per_epoch=len(train_loader))
    
else:
    print(f"Step LR scheduler is configured with {LR_DECAY_STEPS} epochs for each step")
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                   step_size=LR_DECAY_STEPS,
                                                   gamma=0.1)



### Training script

In [ ]:
from sem_seg_utils import train

In [ ]:
history  = train(
    model=model, 
    num_classes=NUM_CLASSES,
    train_loader=train_loader, 
    test_loader=test_loader, 
    dice_loss_weight=1.0,
    ce_loss_weight=1.0,
    optimizer=optimizer,     
    lr_scheduler=lr_scheduler,
    num_epochs=NUM_EPOCHS,
    device=device, 
    model_path=MODEL_PATH,
    ce_k_factor=5.0
)

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model = seg_models.deeplabv3_resnet50(weights=seg_models.DeepLabV3_ResNet50_Weights.DEFAULT)
model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)
# model = seg_models.deeplabv3_mobilenet_v3_large(weights=seg_models.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT)

# Change final layer to 2 classes 
# note that FCN and DeepLab models have different number of channels for this layer (512 vs 256 for DeepLab)
num_channels: int =  model.classifier[4].state_dict()['weight'].shape[1]
model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 

model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'ctyo_bf_fcn_resnet50_2112_dice_ce_loss_1cycle_lrs_2_bs_10_epochs.pt')))

In [ ]:
from sem_seg_utils import mIoU, pixel_accuracy, evaluate

In [ ]:
evaluate(
    model=model, 
    num_classes=NUM_CLASSES,
    criterion=None, 
    data_loader=test_loader, 
    device=device, 
    return_loss=False, 
)

In [ ]:
def predict(model, image, device):
    
    # mean: Final[float] = 0.449 
    # std: Final[float] = 0.226 
    mean: Final[np.ndarray] = np.array([0.485, 0.456, 0.406]) 
    std: Final[np.ndarray] = np.array([0.229, 0.224, 0.225])
    model_input_size: Final[int] = 2112
    
    img_height, img_width = image.shape[:2]

    if max(img_height, img_width) > model_input_size:
        if img_height > img_width:
            img = cv2.resize(image, (int((img_width * model_input_size) / float(img_height)), model_input_size), interpolation=cv2.INTER_AREA)
        else:
            img = cv2.resize(image, (model_input_size, int((img_height * model_input_size) / float(img_width))), interpolation=cv2.INTER_AREA)        
    else:
        img = image
        
    img = (img.astype(float) / 255.0 - mean) / std
    
    image_tensor = F.to_tensor(img).unsqueeze(dim=0).to(device).float()
    
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        output = model(image_tensor)["out"]
        mask = torch.argmax(output, dim=1).squeeze().cpu().numpy().astype(np.uint8)

    if max(img_height, img_width) > model_input_size:
        mask = cv2.resize(mask, (img_width, img_height), interpolation=cv2.INTER_NEAREST)
       
    return mask

In [ ]:
mask = predict(model, test_dataset.dataset[3]['image'], device)

In [ ]:
Image.fromarray(show_sample(3, test_dataset))

In [ ]:
Image.fromarray(mask * 255)

In [ ]:
import time
start = time.time()
for i in range(100):
    mask = predict(model, test_dataset.parsed_json[1]['image'], device)
print(f"Model runtime took {np.round((time.time() - start) * 10, 2)} ms")

In [ ]:
import os
os.listdir('checkpoints')

In [ ]:
# old model, old test set
# Test mean IoU: 0.829 
# Test Accuracy: 0.967

# new model, old test set
# Test mean IoU: 0.827 
# Test Accuracy: 0.968

# old model, new test set (more cytoplasm annotations and images with cages but no cytoplasm) 
# Test mean IoU: 0.878 
# Test Accuracy: 0.954    

# new model, new test set
# Test mean IoU: 0.930 
# Test Accuracy: 0.989

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
cyto_model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)
num_channels: int =  cyto_model.classifier[4].state_dict()['weight'].shape[1]
cyto_model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 
cyto_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'cyto_bf_fcn_resnet50_1024_dice_ce_loss_1cycle_lrs_2_bs_10_epochs.pt')))


nucleus_model = seg_models.fcn_resnet50(weights=seg_models.FCN_ResNet50_Weights.DEFAULT)
num_channels: int =  nucleus_model.classifier[4].state_dict()['weight'].shape[1]
nucleus_model.classifier[4] = torch.nn.Conv2d(num_channels, NUM_CLASSES, kernel_size=(1, 1), stride=(1, 1)) 
nucleus_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'nucleus_bf_fcn_resnet50_1024_dice_ce_loss_1cycle_lrs_2_bs_10_epochs.pt')))

In [ ]:
Z_STACK_PATH = '/home/labuser/workspaces/mehdi/semantic_segmentation/data/20250227_preadipocytes-adhered_4x_caged_z_stack_overlaid_16u_6_class'
src_test_path = os.path.join(Z_STACK_PATH, 'z_stack_images', 'test')
src_train_path = os.path.join(Z_STACK_PATH, 'z_stack_images', 'train')
dest_test_path = os.path.join(Z_STACK_PATH, 'images', 'test')
dest_train_path = os.path.join(Z_STACK_PATH, 'images', 'train')

In [ ]:
# from json_parser import create_overlaid_img
def create_overlaid_img(bf_image: np.ndarray, fl_images_dict: Dict[str, np.ndarray], normalize_fl_image_hist: bool = True) -> np.ndarray:
    """
    Overlay the brightfield image for an FoV with the FL channel images.

    Args:
        bf_image (np.ndarray): Numpy array of the brightfield image for the FoV. 
        fl_images_dict (Dict[str, np.ndarray]): A dictionary with keys as 'red', 'green' and 'blue'
            (the colors to map the corresponding FL image) and values as the numpy array for the FL images. 
            All the images should be of the same size and should only include 1 channel. This dictionary
            may include only a subset of colors. 
        normalize_fl_image_hist (bool, optional): Apply histogram normalization on the FL images before overlaying. 
    Returns:
        numpy array of the overlaid images. 
    """
    overlaid_image: np.ndarray = None
    # colors in RGB format, this order is what we ultimately use for training
    colors: Dict[str, np.array] = {
        "red": np.array([1.0, 0, 0]),
        "green": np.array([0, 1.0, 0]),
        "blue": np.array([0, 0, 1.0])
        
    }

    for k, v in fl_images_dict.items():
        # copy to make sure we are not modifying the input image arrays
        fl_image: np.ndarray = v.copy()
        if len(fl_image.shape) > 2 and fl_image.shape[2] > 1:
            print(f"[WARN] create_overlaid_img expects a 2D image, but a 3D image was passed for FL channel in '{k}'! "
                  f"RGB image will be converted to gray scale before overlaying")
            fl_image = cv2.cvtColor(fl_image, cv2.COLOR_BGR2GRAY)
    
        if normalize_fl_image_hist:
            # create a CLAHE object
            clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(128, 128))
            fl_image = clahe.apply(fl_image)

        # convert back to a 3D and map to the channel
        fl_image = (cv2.cvtColor(fl_image, cv2.COLOR_GRAY2RGB) * colors[k]).astype(np.uint8)
            
        if overlaid_image is None:
            overlaid_image = fl_image
        else:
            # clipping not not needed as we map the images to different channels
            overlaid_image = np.clip(overlaid_image + fl_image, 0, 255)

    if bf_image is None:
        return overlaid_image
    
    if len(bf_image.shape) > 2 and bf_image.shape[2] > 1:
        print("[WARN] create_overlaid_img expects a 2D image, but a 3D image was passed for the brightfield channel! ") 
        print("RGB image will be converted to gray scale before overlaying")
        bf_image = cv2.cvtColor(bf_image, cv2.COLOR_BGR2GRAY)

    bf_image = cv2.cvtColor(bf_image, cv2.COLOR_GRAY2RGB)
    
    if overlaid_image is None:
        overlaid_image = bf_image
    else:
        overlaid_image = np.clip(0.8 * bf_image + 0.2 * overlaid_image, 0, 255).astype(np.uint8)
    return overlaid_image

In [ ]:
src_path = src_test_path
dest_path = dest_test_path

src_images = os.listdir(src_path)

for img_name in src_images:
    img = Image.open(os.path.join(src_path, img_name))
    img = np.array(img) # RGB format, as expected by the model
    nucleus_mask = predict(nucleus_model, img, device)
    cyto_mask = predict(cyto_model, img, device)
    overlaid_img = create_overlaid_img(
        bf_image=img[:, :, 1], 
        fl_images_dict = {'red': nucleus_mask * 255, 'blue': cyto_mask * 255}, 
        normalize_fl_image_hist=False
    ) 
    overlaid_img = cv2.cvtColor(overlaid_img, cv2.COLOR_BGR2RGB)
    cv2.imwrite(os.path.join(dest_path, img_name), overlaid_img)

print(len(os.listdir(dest_path)))